<a href="https://colab.research.google.com/github/afnaayusuf/Tracer/blob/main/Tracer_FM_Build_Model_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. make sure the unpacked repo is the latest zip (the one with the `tiles` command)
!cd /content && unzip -o -q tracer-fm.zip

In [ ]:
# 2. point the config at the real file — keep the space
import json
p = "/content/tracer-fm/configs/pedido_grid.json"
c = json.load(open(p)); c["grid"]["source"] = "/content/PEDIDO 27057.mp4"
json.dump(c, open(p, "w+"), indent=1)

In [ ]:
!pip install -r /content/tracer-fm/requirements.txt -r /content/tracer-fm/requirements-gpu.txt

  Cloning https://github.com/microsoft/MoGe.git to /tmp/pip-req-build-c601zyqk
  Running command git clone --filter=blob:none --quiet https://github.com/microsoft/MoGe.git /tmp/pip-req-build-c601zyqk
  Resolved https://github.com/microsoft/MoGe.git to commit 74fbce054ebed49800de42d0ad0e83495065719a
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 5.1 MB/s eta 0:00:00
  Cloning https://github.com/EasternJournalist/utils3d-moge.git (to revision 62f09d58509485564e24d5d9f6aac9ee9ebc0c37) to /tmp/pip-install-k3yvteqh/utils3d-moge_e8f54cbc578c4196b27efd36aef35fe6
  Running command git clone --filter=blob:none --quiet https://github.com/EasternJournalist/utils3d-moge.git /tmp/pip-install-k3yvteqh/utils3d-moge_e8f54cbc578c4196b27efd36aef35fe6
  Running command git rev-parse -q --verify 'sha^62f09d58509485564e24d5d9f6aac9ee9ebc0c37'
  Running command g

In [ ]:
%%bash
ffmpeg -hide_banner -decoders 2>/dev/null | grep -iE "dav1d|libaom|av1" || echo "NO AV1 DECODER"

 V..... libdav1d             dav1d AV1 decoder by VideoLAN (codec av1)
 V....D libaom-av1           libaom AV1 (codec av1)
 V....D av1                  Alliance for Open Media AV1
 V..... av1_cuvid            Nvidia CUVID AV1 decoder (codec av1)
 V....D av1_qsv              AV1 video (Intel Quick Sync Video acceleration) (codec av1)
 A....D wmav1                Windows Media Audio 1


In [ ]:
%cd /content/tracer-fm
!python -m tracer.cli tiles --config configs/pedido_grid.json --out /content/tiles

/content/tracer-fm
ch7: rect x=2 y=2 w=636 h=356 -> /content/tiles/ch7.png
ch4: rect x=642 y=2 w=636 h=356 -> /content/tiles/ch4.png
ch3: rect x=2 y=362 w=636 h=356 -> /content/tiles/ch3.png
ch5: rect x=642 y=362 w=636 h=356 -> /content/tiles/ch5.png
grid 1280x720 native 28.0 fps, gate step 7, t_ms 0


In [ ]:
import json; p="/content/tracer-fm/configs/pedido_grid.json"; c=json.load(open(p))
c["grid"]["source"]="/content/pedido_h264.mp4"; json.dump(c,open(p,"w"),indent=1)

In [ ]:
%%bash
cd /content
ffmpeg -hide_banner -loglevel error -stats -i "PEDIDO 27057.mp4" \
  -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p -an pedido_h264.mp4
ffprobe -v error -select_streams v:0 -show_entries stream=codec_name,width,height,r_frame_rate,nb_frames -of default=nw=1 pedido_h264.mp4

frame= 3169 fps=1577 q=24.0 size=   15104kB time=00:01:53.10 bitrate=1093.9kbits/s speed=56.3x    
Enter command: <target>|all <time>|-1 <command>[ <argument>]

frame=17719 fps=1610 q=-1.0 Lsize=   79405kB time=00:10:32.71 bitrate=1028.1kbits/s speed=57.5x    


In [ ]:
# 4. only display what exists
import os
from IPython.display import Image, display
for n in ["ch7", "ch4", "ch3", "ch5"]:
    f = f"/content/tiles/{n}.png"
    display(Image(f, width=360)) if os.path.exists(f) else print("missing", f)

missing /content/tiles/ch7.png
missing /content/tiles/ch4.png
missing /content/tiles/ch3.png
missing /content/tiles/ch5.png


In [ ]:
%cd /content/tracer-fm
!python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"
!python -c "import rfdetr, supervision, transformers; from moge.model.v2 import MoGeModel; print('imports ok')"
!python -c "import json; c=json.load(open('configs/pedido_grid.json')); print(c['grid']['source'], c['models']['depth']['model_id'], [e['type'] for e in c['models']['embedders']])"

/content/tracer-fm
2.11.0+cu128 12.8 NVIDIA RTX PRO 6000 Blackwell Server Edition
imports ok
/content/pedido_h264.mp4 Ruicheng/moge-2-vitb-normal ['siglip2', 'dinov3']


In [ ]:
!nvidia-smi --query-gpu=name,driver_version --format=csv
!nvidia-smi | grep "CUDA Version"
!pip list 2>/dev/null | grep -iE "^(torch|torchvision) "
!python -c "import torch; print(torch.__version__, 'cuda:', torch.version.cuda)"

/bin/bash: line 1: nvidia-smi: command not found
/bin/bash: line 1: nvidia-smi: command not found
torch                                 2.11.0+cpu
torchvision                           0.26.0+cpu
2.11.0+cpu cuda: None


In [ ]:
!pip uninstall -y torch torchvision torchaudio
!pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu128
!python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"
!python -c "import torchvision, rfdetr, transformers; from moge.model.v2 import MoGeModel; print('imports ok')"

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 820.3/820.3 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 85.0 MB/s eta 0:00:00
2.11.0+cu128 12.8 NVIDIA RTX PRO 6000 Blackwell Server Edition
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import torchvision, rfdetr, transformers; from moge.model.v2 import MoGeModel; print('imports ok')
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'rfdetr'


In [ ]:
%%bash
cd /content/tracer-fm
pip uninstall -y -q torch torchvision torchaudio
pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu128
python -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"
python -c "import rfdetr, supervision, transformers; from moge.model.v2 import MoGeModel; print('imports ok, transformers', transformers.__version__)"

2.11.0+cu128 12.8 NVIDIA RTX PRO 6000 Blackwell Server Edition


bash: line 1: cd: /content/tracer-fm: No such file or directory
Traceback (most recent call last):
  File "<string>", line 1, in <module>
    import rfdetr, supervision, transformers; from moge.model.v2 import MoGeModel; print('imports ok, transformers', transformers.__version__)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ModuleNotFoundError: No module named 'rfdetr'


CalledProcessError: Command 'b'cd /content/tracer-fm\npip uninstall -y -q torch torchvision torchaudio\npip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu128\npython -c "import torch; print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"\npython -c "import rfdetr, supervision, transformers; from moge.model.v2 import MoGeModel; print(\'imports ok, transformers\', transformers.__version__)"\n'' returned non-zero exit status 1.

In [3]:
%%bash
set -e
# ======================= settings (edit these) =======================
HF_TOKEN=""                          # hf_... token WITH DINOv3 access; empty = drop DINOv3 (SigLIP 2 does appearance)
OSD="2026-09-28 13:05:00"            # clock on the first frame (top-right of /content/tiles/ch7.png after step 5)
TZ_NAME="America/Mexico_City"        # store timezone
TEXT_QUERY="man in a blue shirt"     # appearance text used to pick the entities for the packet
QUESTION="When did the man in the blue shirt arrive at the counter, how long did he stay, and which cameras saw him?"
RUN_QUERY=1                          # 1 = install vLLM, serve, ask;  0 = stop after the packet
QUERY_MODEL="Qwen/Qwen3.5-9B"        # H100 alternative: Qwen/Qwen3.5-27B
# ======================================================================
ZIP=/content/tracer-fm.zip; REPO=/content/tracer-fm; LIB=/content/lib_pedido; TILES=/content/tiles
H264=/content/pedido_h264.mp4; VENV=/content/vllm_env
if [ -n "$HF_TOKEN" ]; then export HF_TOKEN; else unset HF_TOKEN; fi
step(){ echo; echo "================ $1"; }

step "0. repo"
cd /content && { [ -d $REPO ] || unzip -q $ZIP -d /content; } && cd $REPO && ls

step "1. video"
SRC=$(ls /content/PEDIDO*.mp4 2>/dev/null | head -1)
[ -n "$SRC" ] || { echo "no /content/PEDIDO*.mp4 found"; exit 1; }
CODEC=$(ffprobe -v error -select_streams v:0 -show_entries stream=codec_name -of csv=p=0 "$SRC")
echo "source: $SRC   codec: $CODEC"
if [ "$CODEC" = "av1" ]; then
  if [ ! -s $H264 ]; then
    if ffmpeg -hide_banner -decoders 2>/dev/null | grep -qiE "libdav1d|libaom-av1"; then FF=ffmpeg
    else pip install -q imageio-ffmpeg; FF=$(python -c "import imageio_ffmpeg;print(imageio_ffmpeg.get_ffmpeg_exe())"); fi
    echo "transcoding AV1 -> H.264 with $FF"
    "$FF" -hide_banner -loglevel error -stats -i "$SRC" -c:v libx264 -preset veryfast -crf 18 -pix_fmt yuv420p -an $H264
  fi
  VIDEO=$H264
else
  VIDEO="$SRC"
fi
ffprobe -v error -select_streams v:0 -show_entries stream=codec_name,width,height,r_frame_rate,nb_frames -of default=nw=1 "$VIDEO"

step "2. ingest deps"
python -c "import rfdetr, supervision, transformers; from moge.model.v2 import MoGeModel" 2>/dev/null || \
  pip install -q rfdetr supervision "transformers>=4.56" git+https://github.com/microsoft/MoGe.git

step "3. torch must be CUDA (the deps may have swapped it)"
if ! python -c "import torch,sys; sys.exit(0 if torch.cuda.is_available() else 1)" 2>/dev/null; then
  pip uninstall -y -q torch torchvision torchaudio || true
  pip install -q torch torchvision --index-url https://download.pytorch.org/whl/cu128
fi
python -c "import torch; print('torch', torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))"
python -c "import rfdetr, supervision, transformers; from moge.model.v2 import MoGeModel; print('imports ok, transformers', transformers.__version__)"

step "4. config + start time"
python - <<EOF
import json
p="configs/pedido_grid.json"; c=json.load(open(p))
c["grid"]["source"]="$VIDEO"
c["lib"]["root"]="$LIB"
c["models"]["depth"]={"type":"moge2","model_id":"Ruicheng/moge-2-vitl-normal","resolution_level":6}
if not "$HF_TOKEN":
    c["models"]["embedders"]=[e for e in c["models"]["embedders"] if e["type"]!="dinov3"]
    c["tracker"]["appearance_model"]="siglip2"
json.dump(c,open(p,"w"),indent=1)
print("embedders:", [e["type"] for e in c["models"]["embedders"]], "| appearance:", c["tracker"]["appearance_model"], "| depth:", c["models"]["depth"]["model_id"])
EOF
START=$(python -c "from datetime import datetime; from zoneinfo import ZoneInfo; print(int(datetime.fromisoformat('$OSD').replace(tzinfo=ZoneInfo('$TZ_NAME')).timestamp()*1000))")
echo "START_MS $START"

step "5. tile check  (open $TILES/ch7.png .. ch5.png afterwards)"
rm -rf $TILES; python -m tracer.cli tiles --config configs/pedido_grid.json --out $TILES

step "6. ingest"
rm -rf $LIB
python -m tracer.cli ingest --config configs/pedido_grid.json --start-ms $START
python -m tracer.cli inspect --root $LIB

step "7. entities + packet"
python - <<EOF
import json
from tracer.lib.store import LibReader
from tracer.virtualizer.tools import Virtualizer
from tracer.com.adapters import SigLIP2Embedder
rd=LibReader("$LIB")
idx=[l for cam in rd.cams("shop") for l in rd.index("shop",cam)]
assert idx, "lib is empty: the detector found nobody"
T0=min(l["t0_ms"] for l in idx); T1=max(l["t1_ms"] for l in idx)
json.dump({"T0":T0,"T1":T1},open("/content/window.json","w")); print("window", T0, T1, (T1-T0)/1000, "s")
vz=Virtualizer(rd, SigLIP2Embedder(device="cuda"))
hits=vz.find_entities("shop",T0,T1,text="$TEXT_QUERY",top_k=8)
for h in hits: print(" ", h["entity"], h["cam"], "score", h["text_score"], "ticks", h["n_ticks"], "linked", h["linked"])
best={}
for h in hits:
    best.setdefault(h["cam"], h["entity"])
ents=list(best.values())[:2]
json.dump(ents,open("/content/entities.json","w")); print("packet entities:", ents)
EOF
T0=$(python -c "import json;print(json.load(open('/content/window.json'))['T0'])")
T1=$(python -c "import json;print(json.load(open('/content/window.json'))['T1'])")
E=$(python -c "import json;print(' '.join(json.load(open('/content/entities.json'))))")
P=$(python -c "import json;e=json.load(open('/content/entities.json'));print(e[0]+':'+e[1] if len(e)>1 else '')")
python -m tracer.cli packet --root $LIB --key shop --t0 $T0 --t1 $T1 --entities $E ${P:+--pairs $P} --text-model google/siglip2-base-patch16-256

if [ "$RUN_QUERY" = "1" ]; then
step "8. query model (vLLM in its own venv, so it cannot touch the ingest torch)"
[ -x $VENV/bin/vllm ] || { pip install -q uv; uv venv $VENV --python 3.12 -q; uv pip install --python $VENV/bin/python -q vllm; }
PARSER=$($VENV/bin/vllm serve --help 2>/dev/null | python -c '
import re, sys
h = sys.stdin.read()
m = re.search(r"--tool-call-parser.{0,400}?\{([^}]+)\}", h, re.S) or re.search(r"--tool-call-parser.{0,400}?[Pp]ossible choices:\s*([^\n]+)", h, re.S)
choices = [c.strip() for c in m.group(1).split(",")] if m else []
pref = [c for c in choices if "qwen3_5" in c or "qwen35" in c] + [c for c in choices if "hermes" in c]
print(pref[0] if pref else "hermes")
')
echo "tool parser: $PARSER"
if ! curl -s localhost:8000/v1/models | grep -q '"id"'; then
  setsid nohup $VENV/bin/vllm serve $QUERY_MODEL --max-model-len 32768 --gpu-memory-utilization 0.85 \
     --enable-auto-tool-choice --tool-call-parser $PARSER > /content/vllm.log 2>&1 &
fi
for i in $(seq 1 120); do
  curl -s localhost:8000/v1/models | grep -q '"id"' && break
  sleep 15
  [ $((i % 4)) -eq 0 ] && echo "  waiting for vLLM ($((i * 15))s): $(tail -1 /content/vllm.log | cut -c1-140)"
done
curl -s localhost:8000/v1/models | grep -q '"id"' || { echo "vLLM did not come up:"; tail -30 /content/vllm.log; exit 1; }
python -m tracer.cli query --root $LIB --key shop --t0 $T0 --t1 $T1 --llm http://localhost:8000/v1 --model $QUERY_MODEL \
   --text-model google/siglip2-base-patch16-256 --trace "$QUESTION"
fi

echo; echo "DONE   lib=$LIB   window=$T0..$T1   entities=$E   tiles=$TILES   vllm log=/content/vllm.log"

Process is interrupted.


In [5]:
import zipfile, shutil
REPO = "/content/tracer-fm"; ZIP = "/content/tracer-fm.zip"
p = f"{REPO}/tracer/com/adapters.py"
s = open(p).read()
if "def coco_class_names" not in s:
    s = s.replace('''        from rfdetr.util.coco_classes import COCO_CLASSES  # type: ignore
        self.class_names = COCO_CLASSES
''', '''        self.class_names = coco_class_names()
''')
    table = '''# --------------------------------------------------------------------------- detection
_COCO_IDS = {1: "person", 2: "bicycle", 3: "car", 4: "motorcycle", 5: "airplane", 6: "bus", 7: "train", 8: "truck", 9: "boat",
             10: "traffic light", 11: "fire hydrant", 13: "stop sign", 14: "parking meter", 15: "bench", 16: "bird", 17: "cat",
             18: "dog", 19: "horse", 20: "sheep", 21: "cow", 22: "elephant", 23: "bear", 24: "zebra", 25: "giraffe", 27: "backpack",
             28: "umbrella", 31: "handbag", 32: "tie", 33: "suitcase", 34: "frisbee", 35: "skis", 36: "snowboard", 37: "sports ball",
             38: "kite", 39: "baseball bat", 40: "baseball glove", 41: "skateboard", 42: "surfboard", 43: "tennis racket", 44: "bottle",
             46: "wine glass", 47: "cup", 48: "fork", 49: "knife", 50: "spoon", 51: "bowl", 52: "banana", 53: "apple", 54: "sandwich",
             55: "orange", 56: "broccoli", 57: "carrot", 58: "hot dog", 59: "pizza", 60: "donut", 61: "cake", 62: "chair", 63: "couch",
             64: "potted plant", 65: "bed", 67: "dining table", 70: "toilet", 72: "tv", 73: "laptop", 74: "mouse", 75: "remote",
             76: "keyboard", 77: "cell phone", 78: "microwave", 79: "oven", 80: "toaster", 81: "sink", 82: "refrigerator", 84: "book",
             85: "clock", 86: "vase", 87: "scissors", 88: "teddy bear", 89: "hair drier", 90: "toothbrush"}


def coco_class_names():
    """rfdetr moved its class table (rfdetr.util -> rfdetr.utilities in 1.9). Try both, else the standard COCO ids."""
    for mod in ("rfdetr.utilities.coco_classes", "rfdetr.util.coco_classes"):
        try:
            return getattr(__import__(mod, fromlist=["COCO_CLASSES"]), "COCO_CLASSES")
        except Exception:
            continue
    return _COCO_IDS


class RFDETRDetector:'''
    s = s.replace('''# --------------------------------------------------------------------------- detection
class RFDETRDetector:''', table)
    assert "def coco_class_names" in s and "rfdetr.util.coco_classes import" not in s, "patch did not apply"
    open(p, "w").write(s)
    tmp = ZIP + ".tmp"
    with zipfile.ZipFile(ZIP) as zin, zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zout:
        for item in zin.infolist():
            data = s.encode() if item.filename == "tracer-fm/tracer/com/adapters.py" else zin.read(item.filename)
            zout.writestr(item, data)
    shutil.move(tmp, ZIP)
    print("patched adapters.py in repo and zip")
else:
    print("already patched")

patched adapters.py in repo and zip


In [6]:
%cd /content/tracer-fm
import numpy as np, json
from PIL import Image
from tracer.com.adapters import RFDETRDetector
det = RFDETRDetector(size="medium", threshold=0.4, classes=None)
img = np.asarray(Image.open("/content/tiles/ch7.png").convert("RGB"))
for d in det.detect(img): print(d.cls, round(d.score, 2), d.bbox.astype(int).tolist())

/content/tracer-fm
[2026-09-30 06:32:11] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


[2026-09-30 06:32:11] [WARNING] rf-detr - Using a different number of positional encodings than DINOv2, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.
[2026-09-30 06:32:11] [WARNING] rf-detr - Using patch size 16 instead of 14, which means we're not loading DINOv2 backbone weights. This is not a problem if finetuning a pretrained RF-DETR model.


[2026-09-30 06:32:12] [INFO] rf-detr - File /root/.roboflow/models/rf-detr-medium.pth already exists with correct MD5 hash.


[2026-09-30 06:32:13] [WARNING] rf-detr - Model is not optimized for inference. Latency may be higher than expected. For full GPU throughput (e.g. ~8x on T4 via FP16 Tensor Cores), call model.inference(dtype=torch.float16).
/usr/local/lib/python3.13/dist-packages/rfdetr/detr.py:115: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  return torch.from_numpy(image.transpose((2, 0, 1)))


person 0.95 [364, 15, 550, 317]
handbag 0.49 [515, 295, 612, 355]
refrigerator 0.41 [316, 0, 368, 130]


In [8]:
!grep -c "def coco_class_names" /content/tracer-fm/tracer/com/adapters.py
!unzip -p /content/tracer-fm.zip tracer-fm/tracer/com/adapters.py | grep -c "def coco_class_names"

1
1


In [14]:
%%bash
cd /content/tracer-fm
echo "patched lines on disk: $(grep -c '_features(self.model.get_' tracer/com/adapters.py)"
python - <<'EOF'
import numpy as np, torch
from PIL import Image
img = np.asarray(Image.open("/content/tiles/ch7.png").convert("RGB"))
from tracer.com.adapters import SigLIP2Embedder
emb = SigLIP2Embedder(device="cuda")
v = emb.embed([img[15:317, 364:550]]); t = emb.embed_text(["man in a blue shirt", "empty counter"])
print("siglip2 image", v.shape, "text", t.shape, "sims", (v @ t.T).round(3))
from tracer.mdr.adapters import MoGe2Depth
dm = MoGe2Depth("Ruicheng/moge-2-vitl-normal", device="cuda", resolution_level=6)
out = dm.infer(img)
print("depth", out.depth.shape, "median m", float(np.nanmedian(out.depth)), "person region median m", float(np.nanmedian(out.depth[15:317, 364:550])))
print("intrinsics", out.intrinsics)
EOF

patched lines on disk: 2
siglip2 image (1, 768) text (2, 768) sims [[0.104 0.069]]
depth (356, 636) median m 1.571036458015442 person region median m 1.407234787940979
intrinsics Intrinsics(fx=495.8813171386719, fy=495.8813171386719, cx=318.0, cy=178.0, width=636, height=356)


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100%|██████████| 408/408 [00:00<00:00, 15769.08it/s]
/content/tracer-fm/tracer/mdr/adapters.py:22: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  t = torch.from_numpy(np.ascontiguousarray(image)).to(device).float().div(255.0).permute(2, 0, 1)


In [1]:
import zipfile, shutil
REPO = "/content/tracer-fm"; ZIP = "/content/tracer-fm.zip"
p = f"{REPO}/tracer/com/adapters.py"
s = open(p).read()
if "def _features" not in s:
    s = s.replace('''# --------------------------------------------------------------------------- embeddings
class SigLIP2Embedder:''', '''# --------------------------------------------------------------------------- embeddings
def _features(out):
    """transformers < 5 returns a tensor from get_image_features/get_text_features; >= 5 returns a
    ModelOutput. The pooled embedding is the same quantity in both; take it either way."""
    import torch  # noqa
    if torch.is_tensor(out):
        return out
    for name in ("pooler_output", "image_embeds", "text_embeds"):
        v = getattr(out, name, None)
        if v is not None:
            return v
    return out.last_hidden_state[:, 0]


class SigLIP2Embedder:''')
    s = s.replace("            f = self.model.get_image_features(**inp)\n", "            f = _features(self.model.get_image_features(**inp))\n")
    s = s.replace("            f = self.model.get_text_features(**inp)\n", "            f = _features(self.model.get_text_features(**inp))\n")
    assert s.count("_features(self.model.get_") == 2 and "def _features" in s, "patch did not apply"
    open(p, "w").write(s)
    tmp = ZIP + ".tmp"
    with zipfile.ZipFile(ZIP) as zin, zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED) as zout:
        for item in zin.infolist():
            zout.writestr(item, s.encode() if item.filename == "tracer-fm/tracer/com/adapters.py" else zin.read(item.filename))
    shutil.move(tmp, ZIP)
    print("patched SigLIP2 adapter in repo and zip")
else:
    print("already patched")

FileNotFoundError: [Errno 2] No such file or directory: '/content/tracer-fm/tracer/com/adapters.py'

In [4]:
# 1. unzip + patch
!cd /content && unzip -q tracer-fm.zip && python patch_tracer_v2.py /content/tracer-fm

replace tracer-fm/requirements.txt? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/README.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/scripts/colab_pedido.md? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/.gitignore? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/configs/pedido_grid.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/configs/example.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/tests/test_virtualizer.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/tests/test_pipeline_e2e.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/tests/test_agent.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/tests/__init__.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/tests/test_sources.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/tests/test_geometry_sda.py? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace tracer-fm/pyproject.toml? [y]es, [n]

In [6]:
p = "/content/tracer-fm/tracer/tracking/tracker3d.py"
s = open(p).read()
old = "        r, c = _lsa(cost)\n"
new = ("        # scipy raises 'cost matrix is infeasible' when a row/column is all inf (nothing inside the gate for\n"
       "        # some track or detection). Give those pairs a finite ceiling; max_cost filters them out below.\n"
       "        r, c = _lsa(np.where(np.isfinite(cost), cost, 1e6))\n")
if old in s:
    open(p, "w").write(s.replace(old, new)); print("patched tracker3d.py")
else:
    print("already patched" if "1e6" in s else "UNEXPECTED: pattern not found")

patched tracker3d.py


In [7]:
# 2. everything up to the packet (transcode + installs again on a fresh machine: ~10 min, then ~2 min ingest)
!cd /content/tracer-fm && OSD="2026-09-28 13:05:00" bash scripts/run_pedido_colab.sh


================ 0. repo
configs		README.md	      requirements.txt	tests
pyproject.toml	requirements-gpu.txt  scripts		tracer

================ 1. video
source: /content/PEDIDO 27057.mp4   codec: av1
codec_name=h264
width=1280
height=720
r_frame_rate=28/1
nb_frames=6334

================ 2. ingest deps

================ 3. torch must be CUDA (the deps may have swapped it)
torch 2.11.0+cu128 12.8 NVIDIA RTX PRO 6000 Blackwell Server Edition
imports ok, transformers 5.16.1

================ 4. config + start time
embedders: ['siglip2'] | appearance: siglip2 | depth: Ruicheng/moge-2-vitl-normal
START_MS 1790622300000

================ 5. tile check  (open /content/tiles/ch7.png .. ch5.png afterwards)
ch7: rect x=2 y=2 w=636 h=356 -> /content/tiles/ch7.png
ch4: rect x=642 y=2 w=636 h=356 -> /content/tiles/ch4.png
ch3: rect x=2 y=362 w=636 h=356 -> /content/tiles/ch3.png
ch5: rect x=642 y=362 w=636 h=356 -> /content/tiles/ch5.png
grid 1280x720 native 28.0 fps, gate step 7, t_ms 0

========

In [9]:
!cd /content/tracer-fm && VLLM_USE_FLASHINFER_SAMPLER=0 bash scripts/query_pedido_colab.sh

== vLLM venv
vllm 0.30.0 | torch 2.13.0+cu130 | NVIDIA RTX PRO 6000 Blackwell Server Edition
tool parser: hermes
== serve Qwen/Qwen3.5-9B 
Capturing CUDA graphs (FULL): 100%|████
== query
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100% 408/408 [00:00<00:00, 17700.96it/s]
The user is asking about a man in a blue shirt at a counter. I need to:
1. First find entities matching "man in a blue shirt" using find_entities with the text filter
2. Then get their traces to see when they arrived at the counter and how long they stayed
3. Identify which cameras saw them

Let me start by finding entities with the text filter for "man in a blue shirt" or similar. The window is t0_ms=1790622300000 t

In [10]:
!cd /content/tracer-fm && VLLM_USE_FLASHINFER_SAMPLER=0 VLLM_ATTENTION_BACKEND=TRITON_ATTN bash scripts/query_pedido_colab.sh

== vLLM venv
vllm 0.30.0 | torch 2.13.0+cu130 | NVIDIA RTX PRO 6000 Blackwell Server Edition
tool parser: hermes
== serve Qwen/Qwen3.5-9B 
== query
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100% 408/408 [00:00<00:00, 14293.99it/s]
The user is asking about a man in a blue shirt at a counter. I need to:
1. First find entities matching "man in a blue shirt" using the text filter
2. Then get their traces to see when they arrived at the counter and how long they stayed
3. Identify which cameras saw them

Let me start by finding entities with the text filter for "man in a blue shirt" or similar. The window is t0_ms=1790622300000 to t1_ms=1790622526000.

I should use find_entities with the

In [11]:
!python /content/patch_tracer_v3.py /content/tracer-fm

wrote tracer/agent/runner.py
wrote tracer/tracking/tracker3d.py
wrote scripts/query_pedido_colab.sh
wrote scripts/run_pedido_colab.sh
wrote tests/test_agent.py
wrote tests/test_geometry_sda.py
patch v3 applied to /content/tracer-fm


In [12]:
!cd /content/tracer-fm && RESTART_VLLM=1 bash scripts/query_pedido_colab.sh

== vLLM venv
vllm 0.30.0 | torch 2.13.0+cu130 | NVIDIA RTX PRO 6000 Blackwell Server Edition
tool parser: hermes   reasoning parser: none
== serve Qwen/Qwen3.5-9B 
  waiting (60 s): (APIServer pid=109713) INFO 09-30 14:35:36 [hf.py:642] Detected the chat template content format to be 'openai'. You can set `--chat-templat
== query
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100% 408/408 [00:00<00:00, 15031.90it/s]
[recovered 1 tool call(s) from text]
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790622526000, "text": "man in a blue shirt", "top_k": 10})
[recovered 1 tool call(s) from text]
[tool] get_trace({"entity": "ch7-e1", "t0_ms": 1790622300000, "t1_ms": 1790622526000})


In [13]:
!cd /content/tracer-fm && QUESTION="Did anyone other than the man in the blue shirt appear, and when?" bash scripts/query_pedido_colab.sh

== vLLM venv
vllm 0.30.0 | torch 2.13.0+cu130 | NVIDIA RTX PRO 6000 Blackwell Server Edition
tool parser: hermes   reasoning parser: none
== serve Qwen/Qwen3.5-9B 
== query
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100% 408/408 [00:00<00:00, 14381.56it/s]
[recovered 2 tool call(s) from text]
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790622526000, "text": "man in the blue shirt", "top_k": 10})
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790622526000, "top_k": 20})
The search for "man in the blue shirt" returned several entities, but none have a title that matches "man in the blue shirt" - they all have title "?". The text_score for ch3-e1 is 0.1, which is th

In [14]:
!cd /content/tracer-fm && QUESTION="What else you could see on the counter..?" bash scripts/query_pedido_colab.sh

== vLLM venv
vllm 0.30.0 | torch 2.13.0+cu130 | NVIDIA RTX PRO 6000 Blackwell Server Edition
tool parser: hermes   reasoning parser: none
== serve Qwen/Qwen3.5-9B 
== query
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100% 408/408 [00:00<00:00, 14663.76it/s]
[recovered 1 tool call(s) from text]
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790622526000, "top_k": 20})
[recovered 1 tool call(s) from text]
[tool] select_keyframes({"entities": ["ch3-e1", "ch5-e1", "ch7-e1", "ch4-e3", "ch4-e5", "ch3-e2", "ch3-e3", "ch7-e3", "ch4-e6"], "t0_ms": 1790622300000, "t1_ms": 1790622526000, "budget": 12})
Based on the keyframe images from the counter area, I can see:

**On the counter:**
-

In [15]:
!cd /content/tracer-fm && QUESTION="What are the other things that you see on the desk..?" bash scripts/query_pedido_colab.sh

== vLLM venv
vllm 0.30.0 | torch 2.13.0+cu130 | NVIDIA RTX PRO 6000 Blackwell Server Edition
tool parser: hermes   reasoning parser: none
== serve Qwen/Qwen3.5-9B 
== query
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100% 408/408 [00:00<00:00, 15348.87it/s]
[recovered 1 tool call(s) from text]
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790622526000, "cls": "shop", "top_k": 20})
[recovered 1 tool call(s) from text]
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790622526000, "top_k": 30})
[recovered 1 tool call(s) from text]
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790622526000, "text": "desk", "top_k": 20})
[recovered 1 tool call(s) from text]
[too

In [9]:
# 1. bring the repo to current state
!python /content/patch_tracer_all.py /content/tracer-fm

wrote tracer/com/adapters.py
wrote tracer/config.py
wrote tracer/lib/motion.py
wrote tracer/lib/keyframes.py
wrote tracer/virtualizer/tools.py
wrote tracer/identity/linker.py
wrote tracer/tracking/tracker3d.py
wrote tracer/agent/runner.py
wrote tests/test_motion.py
wrote tests/test_pipeline_e2e.py
wrote tests/test_virtualizer.py
wrote tests/test_agent.py
wrote tests/test_geometry_sda.py
wrote scripts/run_pedido_colab.sh
wrote scripts/query_pedido_colab.sh
cumulative patch applied to /content/tracer-fm - 15 files


In [10]:
# 2. transcode + installs + torch fix + ingest + packet (~12 min on a fresh machine, most of it downloads)
!cd /content/tracer-fm && OSD="2026-09-28 13:05:00" bash scripts/run_pedido_colab.sh


================ 0. repo
configs		README.md	      requirements.txt	tests
pyproject.toml	requirements-gpu.txt  scripts		tracer

================ 1. video
source: /content/PEDIDO 27057.mp4   codec: av1
transcoding AV1 -> H.264 with ffmpeg
frame=118118 fps=1519 q=-1.0 Lsize=  627278kB time=01:10:18.39 bitrate=1218.2kbits/s speed=54.3x    
codec_name=h264
width=1280
height=720
r_frame_rate=28/1
nb_frames=118118

================ 2. ingest deps
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.9/79.9 kB 4.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to bui

In [11]:
# 3. now the lib exists
!cd /content/tracer-fm && python -m tracer.cli inspect --root /content/lib_pedido

{
 "detector": "rfdetr",
 "depth": "moge2",
 "embedders": {
  "siglip2": 768
 },
 "face": null,
 "models": {
  "device": "cuda",
  "detector": {
   "type": "rfdetr",
   "size": "medium",
   "seg": false,
   "threshold": 0.4,
   "classes": [
    "person"
   ]
  },
  "embedders": [
   {
    "type": "siglip2",
    "model_id": "google/siglip2-base-patch16-256"
   }
  ],
  "face": null,
  "depth": {
   "type": "moge2",
   "model_id": "Ruicheng/moge-2-vitl-normal",
   "resolution_level": 6
  },
  "gallery": null
 },
 "vector_dtype": "float16",
 "segment_s": 60.0
}
calib: {}
shop/ch3: 127 index lines, 58 entities
   ch3-e1       person   n=   537 1790622300000..1790622434000 kf=15 titles={'?': 537}
   ch3-e10      person   n=   554 1790623024250..1790623162500 kf=25 titles={'?': 554}
   ch3-e11      person   n=    92 1790623162750..1790623185500 kf=3 titles={'?': 92}
   ch3-e12      person   n=   183 1790623193750..1790623239250 kf=9 titles={'?': 183}
   ch3-e13      person   n=   172 1790623

In [14]:
# 4. query (installs vLLM into its venv again on a fresh machine, then ~1 min to serve)
!cd /content/tracer-fm && QUESTION="Did anyone other than the man in the blue shirt appear, and what are the objects on the desk?" bash scripts/query_pedido_colab.sh

== vLLM venv
vllm 0.30.0 | torch 2.13.0+cu130 | NVIDIA RTX PRO 6000 Blackwell Server Edition
tool parser: hermes   reasoning parser: none
== serve Qwen/Qwen3.5-9B 
== query
[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.
Loading weights: 100% 408/408 [00:00<00:00, 14459.33it/s]
[recovered 1 tool call(s) from text]
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790626516750, "text": "man in the blue shirt"})
[recovered 1 tool call(s) from text]
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790626516750, "text": "desk"})
[recovered 1 tool call(s) from text]
[tool] find_entities({"t0_ms": 1790622300000, "t1_ms": 1790626516750, "text": "objects on the desk"})
[recovered 1 tool call(s) from 

In [1]:
!cd /content && unzip -o -q tracer-fm.zip

In [2]:
!python /content/patch_tracer_all.py /content/tracer-fm

wrote scripts/query_pedido_colab.sh
wrote scripts/run_pedido_colab.sh
wrote tests/test_agent.py
wrote tests/test_geometry_sda.py
wrote tests/test_identity.py
wrote tests/test_motion.py
wrote tests/test_pipeline_e2e.py
wrote tests/test_virtualizer.py
wrote tracer/agent/runner.py
wrote tracer/agent/tools_schema.py
wrote tracer/com/adapters.py
wrote tracer/com/base.py
wrote tracer/config.py
wrote tracer/factory.py
wrote tracer/identity/linker.py
wrote tracer/lib/keyframes.py
wrote tracer/lib/motion.py
wrote tracer/tracking/tracker3d.py
wrote tracer/virtualizer/identity.py
wrote tracer/virtualizer/tools.py
cumulative patch rev 2 applied to /content/tracer-fm - 20 files


In [4]:
!cd /content/tracer-fm && python - <<'EOF'

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `EOF')


In [5]:
import numpy as np
from PIL import Image
from tracer.com.adapters import OWLv2Detector
det = OWLv2Detector(["cardboard box", "package", "plastic bag"], threshold=0.2)
img = np.asarray(Image.open("/content/tiles/ch7.png").convert("RGB"))
for d in det.detect(img): print(d.cls, round(d.score, 2), d.bbox.astype(int).tolist())
EOF

ModuleNotFoundError: No module named 'tracer'